# AI-Powered Travel Itinerary Generator

## Tourism and Hospitality Project

This notebook implements a comprehensive travel assistant agent that generates personalized travel itineraries using:
- **LLM Fine-Tuning** on travel guides and itineraries
- **RAG (Retrieval Augmented Generation)** for real-time location information
- **Intelligent Agent** for custom itinerary creation

### Features:
1. Fine-tuned language model on travel content
2. Vector database for destinations, hotels, and attractions
3. Personalized itinerary generation based on user preferences
4. Real-time information retrieval
5. Activity recommendations

**Platform:** Google Colab Compatible

---

## 1. Setup and Installation

Install required libraries for LLM, RAG, and agent frameworks.

In [ ]:
%%capture
!pip install -q transformers==4.36.0
!pip install -q datasets==2.16.0
!pip install -q accelerate==0.25.0
!pip install -q peft==0.7.1
!pip install -q bitsandbytes==0.41.3
!pip install -q trl==0.7.10
!pip install -q langchain==0.1.0
!pip install -q langchain-community==0.0.10
!pip install -q chromadb==0.4.22
!pip install -q sentence-transformers==2.2.2
!pip install -q openai==1.6.1
!pip install -q tiktoken==0.5.2
!pip install -q faiss-cpu==1.7.4
!pip install -q pypdf==3.17.4
!pip install -q python-dotenv==1.0.0

print("✓ All libraries installed successfully!")

In [ ]:
# Import required libraries
import os
import json
import random
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from typing import List, Dict, Optional
import warnings
warnings.filterwarnings('ignore')

# Transformers and ML
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    pipeline,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset

# LangChain and RAG
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma, FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain.chains import RetrievalQA
from langchain.llms import HuggingFacePipeline

print("✓ All imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Data Collection and Preparation

Create comprehensive datasets for:
- Travel destinations
- Hotels and accommodations
- Attractions and activities
- Travel guides and itineraries
- Reviews and recommendations

In [ ]:
# Sample Travel Destinations Database
destinations_data = [
    {
        "city": "Paris",
        "country": "France",
        "description": "The City of Light, famous for the Eiffel Tower, Louvre Museum, and exquisite cuisine. Perfect for art lovers, history enthusiasts, and romantic getaways.",
        "best_season": "Spring (April-June)",
        "avg_budget_per_day": 150,
        "tags": ["culture", "art", "romance", "history", "food"],
        "recommended_days": 5
    },
    {
        "city": "Tokyo",
        "country": "Japan",
        "description": "A fascinating blend of ancient traditions and cutting-edge technology. Experience temples, gardens, anime culture, and world-class dining.",
        "best_season": "Spring (March-May) for cherry blossoms",
        "avg_budget_per_day": 120,
        "tags": ["culture", "technology", "food", "temples", "shopping"],
        "recommended_days": 7
    },
    {
        "city": "Bali",
        "country": "Indonesia",
        "description": "Tropical paradise with beautiful beaches, rice terraces, and spiritual temples. Ideal for relaxation, yoga retreats, and water sports.",
        "best_season": "April-October",
        "avg_budget_per_day": 70,
        "tags": ["beach", "nature", "adventure", "wellness", "culture"],
        "recommended_days": 6
    },
    {
        "city": "New York",
        "country": "USA",
        "description": "The city that never sleeps. Experience Broadway shows, world-class museums, diverse neighborhoods, and iconic landmarks like the Statue of Liberty.",
        "best_season": "Fall (September-November)",
        "avg_budget_per_day": 200,
        "tags": ["urban", "culture", "entertainment", "shopping", "food"],
        "recommended_days": 5
    },
    {
        "city": "Rome",
        "country": "Italy",
        "description": "The Eternal City with over 2,500 years of history. Explore the Colosseum, Vatican, and indulge in authentic Italian cuisine.",
        "best_season": "Spring (April-June)",
        "avg_budget_per_day": 130,
        "tags": ["history", "culture", "food", "architecture", "art"],
        "recommended_days": 4
    },
    {
        "city": "Dubai",
        "country": "UAE",
        "description": "Futuristic city with luxury shopping, ultramodern architecture, and vibrant nightlife. Home to the Burj Khalifa and stunning desert landscapes.",
        "best_season": "November-March",
        "avg_budget_per_day": 180,
        "tags": ["luxury", "shopping", "modern", "adventure", "beach"],
        "recommended_days": 4
    },
    {
        "city": "Barcelona",
        "country": "Spain",
        "description": "Vibrant coastal city famous for Gaudi's architecture, beautiful beaches, and lively tapas culture.",
        "best_season": "May-June, September-October",
        "avg_budget_per_day": 110,
        "tags": ["beach", "architecture", "culture", "food", "nightlife"],
        "recommended_days": 4
    },
    {
        "city": "Maldives",
        "country": "Maldives",
        "description": "Tropical paradise with crystal-clear waters, overwater bungalows, and world-class diving spots.",
        "best_season": "November-April",
        "avg_budget_per_day": 300,
        "tags": ["beach", "luxury", "honeymoon", "diving", "relaxation"],
        "recommended_days": 5
    }
]

# Hotels Database
hotels_data = [
    {"city": "Paris", "name": "Hotel de la Paix", "category": "Mid-range", "price_per_night": 150, "rating": 4.3, "amenities": ["WiFi", "Breakfast", "City View"]},
    {"city": "Paris", "name": "Le Grand Hotel", "category": "Luxury", "price_per_night": 350, "rating": 4.8, "amenities": ["Spa", "Pool", "Fine Dining", "WiFi"]},
    {"city": "Tokyo", "name": "Shibuya Business Hotel", "category": "Budget", "price_per_night": 80, "rating": 4.0, "amenities": ["WiFi", "Breakfast"]},
    {"city": "Tokyo", "name": "Imperial Tokyo Resort", "category": "Luxury", "price_per_night": 280, "rating": 4.7, "amenities": ["Spa", "Traditional Bath", "Fine Dining"]},
    {"city": "Bali", "name": "Beachfront Villa Resort", "category": "Luxury", "price_per_night": 200, "rating": 4.6, "amenities": ["Pool", "Beach Access", "Spa", "Yoga"]},
    {"city": "Bali", "name": "Ubud Eco Lodge", "category": "Mid-range", "price_per_night": 60, "rating": 4.4, "amenities": ["Nature View", "Breakfast", "WiFi"]},
    {"city": "New York", "name": "Manhattan Plaza Hotel", "category": "Mid-range", "price_per_night": 220, "rating": 4.2, "amenities": ["WiFi", "Gym", "Central Location"]},
    {"city": "Rome", "name": "Vatican View Suites", "category": "Mid-range", "price_per_night": 140, "rating": 4.5, "amenities": ["Rooftop Terrace", "WiFi", "Breakfast"]},
    {"city": "Dubai", "name": "Burj Luxury Residences", "category": "Luxury", "price_per_night": 400, "rating": 4.9, "amenities": ["Infinity Pool", "Spa", "Fine Dining", "Private Beach"]}
]

# Attractions Database
attractions_data = [
    {"city": "Paris", "name": "Eiffel Tower", "category": "Landmark", "duration_hours": 2, "price": 25, "rating": 4.7},
    {"city": "Paris", "name": "Louvre Museum", "category": "Museum", "duration_hours": 4, "price": 17, "rating": 4.8},
    {"city": "Paris", "name": "Seine River Cruise", "category": "Activity", "duration_hours": 2, "price": 15, "rating": 4.5},
    {"city": "Tokyo", "name": "Senso-ji Temple", "category": "Temple", "duration_hours": 2, "price": 0, "rating": 4.6},
    {"city": "Tokyo", "name": "Tokyo Skytree", "category": "Landmark", "duration_hours": 2, "price": 20, "rating": 4.5},
    {"city": "Tokyo", "name": "TeamLab Borderless", "category": "Museum", "duration_hours": 3, "price": 32, "rating": 4.8},
    {"city": "Bali", "name": "Tegallalang Rice Terraces", "category": "Nature", "duration_hours": 3, "price": 5, "rating": 4.6},
    {"city": "Bali", "name": "Tanah Lot Temple", "category": "Temple", "duration_hours": 2, "price": 3, "rating": 4.5},
    {"city": "Bali", "name": "Surfing Lesson at Kuta Beach", "category": "Activity", "duration_hours": 3, "price": 40, "rating": 4.7},
    {"city": "New York", "name": "Statue of Liberty", "category": "Landmark", "duration_hours": 4, "price": 23, "rating": 4.6},
    {"city": "New York", "name": "Central Park", "category": "Nature", "duration_hours": 3, "price": 0, "rating": 4.7},
    {"city": "New York", "name": "Broadway Show", "category": "Entertainment", "duration_hours": 3, "price": 150, "rating": 4.9},
    {"city": "Rome", "name": "Colosseum", "category": "Landmark", "duration_hours": 3, "price": 16, "rating": 4.8},
    {"city": "Rome", "name": "Vatican Museums", "category": "Museum", "duration_hours": 4, "price": 17, "rating": 4.7},
    {"city": "Rome", "name": "Trevi Fountain", "category": "Landmark", "duration_hours": 1, "price": 0, "rating": 4.6},
    {"city": "Dubai", "name": "Burj Khalifa", "category": "Landmark", "duration_hours": 2, "price": 40, "rating": 4.8},
    {"city": "Dubai", "name": "Desert Safari", "category": "Adventure", "duration_hours": 6, "price": 70, "rating": 4.7},
    {"city": "Dubai", "name": "Dubai Mall", "category": "Shopping", "duration_hours": 4, "price": 0, "rating": 4.5}
]

# Create DataFrames
df_destinations = pd.DataFrame(destinations_data)
df_hotels = pd.DataFrame(hotels_data)
df_attractions = pd.DataFrame(attractions_data)

print(f"✓ Created {len(df_destinations)} destinations")
print(f"✓ Created {len(df_hotels)} hotels")
print(f"✓ Created {len(df_attractions)} attractions")

# Display sample data
print("\n=== Sample Destination ===")
print(df_destinations.head(2))

In [ ]:
# Create Training Dataset for LLM Fine-tuning
# Generate travel guide texts and itinerary examples

training_examples = [
    {
        "instruction": "Create a 5-day romantic itinerary for Paris",
        "response": """Day 1: Arrival and Montmartre
- Check into your hotel in the Latin Quarter
- Evening walk through Montmartre, visit Sacré-Cœur
- Dinner at a traditional French bistro

Day 2: Classic Paris
- Morning visit to the Eiffel Tower (arrive early to avoid crowds)
- Lunch at a café with Eiffel Tower views
- Afternoon Seine River cruise
- Evening stroll through Champs-Élysées

Day 3: Art and Culture
- Full morning at the Louvre Museum
- Lunch in Le Marais district
- Afternoon at Musée d'Orsay
- Evening jazz club in Saint-Germain-des-Prés

Day 4: Versailles Day Trip
- Morning train to Palace of Versailles
- Explore the palace and gardens
- Return to Paris for dinner in the Latin Quarter

Day 5: Final Day
- Morning at Notre-Dame Cathedral area
- Shopping on Rue de Rivoli
- Farewell dinner at a Michelin-starred restaurant
"""
    },
    {
        "instruction": "Suggest activities for adventure travelers in Bali",
        "response": """For adventure seekers in Bali, I recommend:

1. Surfing at Uluwatu or Canggu Beach - Perfect waves for all levels
2. Mount Batur Sunrise Trek - 2-hour hike to catch stunning sunrise views
3. White Water Rafting on Ayung River - Class II-III rapids through jungle
4. Scuba Diving at Tulamben - Explore the USAT Liberty shipwreck
5. Canyoning at Aling-Aling Waterfall - Rappelling and cliff jumping
6. ATV Ride through Rice Terraces - Off-road adventure in Ubud
7. Paragliding in Nusa Dua - Aerial views of Bali's coastline

Budget: $50-100 per activity
Best season: April-October for outdoor activities
"""
    },
    {
        "instruction": "What are the must-visit attractions in Tokyo?",
        "response": """Tokyo's must-visit attractions include:

1. Senso-ji Temple (Asakusa) - Tokyo's oldest temple with vibrant atmosphere
2. Tokyo Skytree - Tallest structure in Japan with panoramic views
3. Meiji Shrine - Peaceful Shinto shrine in forested grounds
4. Shibuya Crossing - World's busiest pedestrian crossing
5. Tsukiji Outer Market - Fresh seafood and Japanese street food
6. teamLab Borderless - Immersive digital art museum
7. Imperial Palace East Gardens - Beautiful traditional gardens
8. Harajuku - Youth culture and fashion district
9. Akihabara - Electronics and anime culture hub
10. Tokyo DisneySea - Unique Disney theme park

Recommended: Get a JR Pass for unlimited train travel.
"""
    },
    {
        "instruction": "Plan a budget-friendly 3-day trip to Rome",
        "response": """3-Day Budget Rome Itinerary (€50-70/day):

Day 1: Ancient Rome
- Morning: Colosseum and Roman Forum (combined ticket €16)
- Lunch: Pizza al taglio (€5-8)
- Afternoon: Free visit to Trevi Fountain and Spanish Steps
- Dinner: Trastevere neighborhood trattorias (€15-20)

Day 2: Vatican City
- Early morning: Vatican Museums and Sistine Chapel (€17)
- Free: St. Peter's Basilica and Square
- Lunch: Supplì and street food near Vatican (€8-10)
- Afternoon: Walk along Tiber River, visit Castel Sant'Angelo (€14)
- Dinner: Aperitivo hour for budget dinner (€10-15)

Day 3: Hidden Gems
- Morning: Free walk through Villa Borghese gardens
- Lunch: Campo de' Fiori market picnic (€10)
- Afternoon: Free visit to Pantheon and surrounding piazzas
- Evening: Gelato at Giolitti (€3-5)

Money-saving tips:
- Stay in hostels or budget B&Bs (€25-40/night)
- Buy Roma Pass for transport and museum discounts
- Eat like locals: breakfast at bars, pizza al taglio for lunch
- Many churches and fountains are free to visit
"""
    },
    {
        "instruction": "Best time to visit Maldives and what to expect",
        "response": """Best Time to Visit Maldives:

Peak Season (November-April):
- Dry season with minimal rainfall
- Temperature: 25-30°C (77-86°F)
- Best for: Diving, snorkeling, water sports
- Drawback: Higher prices, more tourists

Shoulder Season (May, October):
- Good weather with occasional showers
- Better deals on resorts
- Still excellent for water activities

Low Season (June-September):
- Monsoon season with more rain
- Significant discounts (30-50% off)
- Rougher seas, but still swimmable

What to Expect:
- Luxury overwater bungalows and private villas
- World-class diving with manta rays and whale sharks
- All-inclusive resort packages
- Water activities: snorkeling, kayaking, paddleboarding
- Spa treatments and wellness retreats
- Limited cultural experiences (most resorts are on private islands)

Budget: $200-500+ per day depending on resort level
Recommended stay: 4-7 days
"""
    },
    {
        "instruction": "Create a family-friendly itinerary for Dubai",
        "response": """4-Day Family Dubai Itinerary:

Day 1: Modern Dubai
- Morning: Burj Khalifa observation deck
- Afternoon: Dubai Mall (Dubai Aquarium, VR Park)
- Evening: Dubai Fountain show and dinner

Day 2: Theme Parks
- Full day at IMG Worlds of Adventure or Legoland Dubai
- These parks offer rides suitable for all ages

Day 3: Beach and Culture
- Morning: La Mer Beach or Jumeirah Beach
- Afternoon: Dubai Museum and Gold Souk
- Evening: Desert Safari with camel rides and BBQ dinner

Day 4: Waterparks
- Aquaventure Waterpark at Atlantis The Palm
- Lost Chambers Aquarium
- Evening: Dubai Marina walk

Family Tips:
- Stay in hotels with kids' clubs (Atlantis, JA Resort)
- Use Dubai Metro for easy transportation
- Book skip-the-line tickets online
- Pack sunscreen and stay hydrated
- Modest dress code for cultural sites

Budget: $300-500/day for family of 4
"""
    }
]

# Add more contextual training data
for dest in destinations_data:
    training_examples.append({
        "instruction": f"Tell me about {dest['city']}, {dest['country']}",
        "response": f"""{dest['city']}, {dest['country']}\n\n{dest['description']}\n\nBest time to visit: {dest['best_season']}\nAverage daily budget: ${dest['avg_budget_per_day']}\nRecommended duration: {dest['recommended_days']} days\nPerfect for: {', '.join(dest['tags'])}"""
    })

print(f"✓ Created {len(training_examples)} training examples")
print("\n=== Sample Training Example ===")
print(f"Instruction: {training_examples[0]['instruction']}")
print(f"Response: {training_examples[0]['response'][:200]}...")

## 3. LLM Fine-Tuning on Travel Data

Fine-tune a language model on travel guides and itineraries using Parameter-Efficient Fine-Tuning (PEFT) with LoRA.

In [ ]:
# Load base model for fine-tuning
# Using a smaller model suitable for Colab (GPT-2 or small Llama-like model)

MODEL_NAME = "gpt2-medium"  # Alternative: "facebook/opt-350m" or "google/flan-t5-base"

print(f"Loading model: {MODEL_NAME}")

# Configure quantization for memory efficiency (if using larger models)
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16
# )

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    # quantization_config=bnb_config  # Uncomment for 4-bit quantization
)

print(f"✓ Model loaded successfully")
print(f"Model parameters: {model.num_parameters():,}")

In [ ]:
# Prepare training dataset
def format_instruction(example):
    """Format training examples into prompt-response pairs"""
    text = f"""### Instruction:
{example['instruction']}

### Response:
{example['response']}"""
    return {"text": text}

# Convert to HuggingFace Dataset
train_dataset = Dataset.from_list(training_examples)
train_dataset = train_dataset.map(format_instruction)

# Tokenize the dataset
def tokenize_function(examples):
    result = tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )
    result["labels"] = result["input_ids"].copy()
    return result

tokenized_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

print(f"✓ Prepared {len(tokenized_dataset)} training samples")

In [ ]:
# Configure LoRA for efficient fine-tuning
lora_config = LoraConfig(
    r=16,  # Rank of the update matrices
    lora_alpha=32,  # LoRA scaling factor
    target_modules=["c_attn"],  # For GPT-2, use ["q_proj", "v_proj"] for Llama
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Prepare model for k-bit training (if using quantization)
# model = prepare_model_for_kbit_training(model)

# Apply LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("✓ LoRA configuration applied")

In [ ]:
# Configure training arguments
training_args = TrainingArguments(
    output_dir="./travel-llm-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=torch.cuda.is_available(),
    logging_steps=10,
    save_strategy="epoch",
    warmup_steps=50,
    report_to="none"
)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer
)

# Start training
print("Starting training...")
print("This may take 10-30 minutes depending on your hardware.\n")

trainer.train()

print("\n✓ Training completed!")

# Save the fine-tuned model
model.save_pretrained("./travel-llm-finetuned/final_model")
tokenizer.save_pretrained("./travel-llm-finetuned/final_model")

print("✓ Model saved successfully")

In [ ]:
# Test the fine-tuned model
def generate_response(instruction, max_length=300):
    """Generate response using fine-tuned model"""
    prompt = f"""### Instruction:
{instruction}

### Response:
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only the response part
    response = response.split("### Response:")[-1].strip()
    return response

# Test with sample queries
test_queries = [
    "What are the best attractions in Tokyo?",
    "Suggest a romantic destination in Europe",
    "Plan a 3-day beach vacation"
]

print("=== Testing Fine-tuned Model ===")
for query in test_queries:
    print(f"\n🔍 Query: {query}")
    response = generate_response(query)
    print(f"🤖 Response: {response[:200]}...")
    print("-" * 80)

## 4. RAG System for Real-Time Information

Implement Retrieval Augmented Generation (RAG) to fetch relevant information about destinations, hotels, and attractions.

In [ ]:
# Create document corpus for RAG
documents = []

# Add destination documents
for dest in destinations_data:
    doc_text = f"""Destination: {dest['city']}, {dest['country']}
Description: {dest['description']}
Best Season: {dest['best_season']}
Average Budget per Day: ${dest['avg_budget_per_day']}
Recommended Days: {dest['recommended_days']}
Tags: {', '.join(dest['tags'])}
"""
    documents.append(Document(
        page_content=doc_text,
        metadata={"type": "destination", "city": dest['city'], "country": dest['country']}
    ))

# Add hotel documents
for hotel in hotels_data:
    doc_text = f"""Hotel: {hotel['name']} in {hotel['city']}
Category: {hotel['category']}
Price per Night: ${hotel['price_per_night']}
Rating: {hotel['rating']}/5.0
Amenities: {', '.join(hotel['amenities'])}
"""
    documents.append(Document(
        page_content=doc_text,
        metadata={"type": "hotel", "city": hotel['city'], "name": hotel['name']}
    ))

# Add attraction documents
for attr in attractions_data:
    doc_text = f"""Attraction: {attr['name']} in {attr['city']}
Category: {attr['category']}
Duration: {attr['duration_hours']} hours
Price: ${attr['price']}
Rating: {attr['rating']}/5.0
"""
    documents.append(Document(
        page_content=doc_text,
        metadata={"type": "attraction", "city": attr['city'], "name": attr['name']}
    ))

print(f"✓ Created {len(documents)} documents for RAG system")

In [ ]:
# Create embeddings and vector store
print("Creating embeddings... (this may take a minute)")

# Use sentence transformers for embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'}
)

# Create FAISS vector store
vectorstore = FAISS.from_documents(documents, embeddings)

print("✓ Vector store created successfully")

# Test retrieval
test_query = "Find me luxury hotels in Paris"
results = vectorstore.similarity_search(test_query, k=3)

print(f"\n=== Testing RAG Retrieval ===")
print(f"Query: {test_query}\n")
for i, doc in enumerate(results, 1):
    print(f"Result {i}:")
    print(doc.page_content)
    print("-" * 60)

In [ ]:
# Create retrieval function for the agent
def retrieve_information(query: str, k: int = 5) -> List[Dict]:
    """Retrieve relevant information from vector store"""
    results = vectorstore.similarity_search(query, k=k)
    
    retrieved_info = []
    for doc in results:
        retrieved_info.append({
            "content": doc.page_content,
            "metadata": doc.metadata
        })
    
    return retrieved_info

# Test the retrieval function
test_queries = [
    "beach destinations for relaxation",
    "budget accommodation in Tokyo",
    "cultural attractions in Rome"
]

print("=== Testing Retrieval Function ===")
for query in test_queries:
    print(f"\n🔍 Query: {query}")
    results = retrieve_information(query, k=2)
    for i, result in enumerate(results, 1):
        print(f"  {i}. {result['metadata']['type']}: {result['content'][:100]}...")
    print("-" * 80)

## 5. Travel Itinerary Agent

Build an intelligent agent that combines the fine-tuned LLM with RAG to create personalized travel itineraries.

In [ ]:
class TravelItineraryAgent:
    """AI Travel Agent for generating personalized itineraries"""
    
    def __init__(self, model, tokenizer, vectorstore, df_destinations, df_hotels, df_attractions):
        self.model = model
        self.tokenizer = tokenizer
        self.vectorstore = vectorstore
        self.df_destinations = df_destinations
        self.df_hotels = df_hotels
        self.df_attractions = df_attractions
    
    def retrieve_context(self, query: str, k: int = 5) -> str:
        """Retrieve relevant context from vector store"""
        results = self.vectorstore.similarity_search(query, k=k)
        context = "\n\n".join([doc.page_content for doc in results])
        return context
    
    def find_destinations(self, preferences: Dict) -> List[Dict]:
        """Find destinations matching user preferences"""
        df = self.df_destinations.copy()
        
        # Filter by budget
        if 'max_budget' in preferences:
            df = df[df['avg_budget_per_day'] <= preferences['max_budget']]
        
        # Filter by tags/interests
        if 'interests' in preferences:
            interests = [i.lower() for i in preferences['interests']]
            df['match_score'] = df['tags'].apply(
                lambda tags: len(set(tags) & set(interests))
            )
            df = df[df['match_score'] > 0].sort_values('match_score', ascending=False)
        
        return df.head(3).to_dict('records')
    
    def find_hotels(self, city: str, budget_category: str = "Mid-range") -> List[Dict]:
        """Find hotels in specified city"""
        df = self.df_hotels[self.df_hotels['city'] == city]
        
        if budget_category:
            df = df[df['category'] == budget_category]
        
        return df.sort_values('rating', ascending=False).head(2).to_dict('records')
    
    def find_attractions(self, city: str, days: int = 3) -> List[Dict]:
        """Find top attractions in city"""
        df = self.df_attractions[self.df_attractions['city'] == city]
        df = df.sort_values('rating', ascending=False)
        
        # Select attractions based on number of days
        num_attractions = min(days * 2, len(df))
        return df.head(num_attractions).to_dict('records')
    
    def generate_itinerary(self, user_preferences: Dict) -> Dict:
        """Generate complete personalized itinerary"""
        # Find matching destinations
        destinations = self.find_destinations(user_preferences)
        
        if not destinations:
            return {"error": "No destinations found matching your preferences"}
        
        # Select primary destination
        primary_dest = destinations[0]
        city = primary_dest['city']
        days = user_preferences.get('days', primary_dest['recommended_days'])
        
        # Find hotels and attractions
        budget_category = self._get_budget_category(user_preferences.get('max_budget', 150))
        hotels = self.find_hotels(city, budget_category)
        attractions = self.find_attractions(city, days)
        
        # Retrieve additional context
        query = f"Travel guide for {city} with {', '.join(user_preferences.get('interests', []))}"
        context = self.retrieve_context(query)
        
        # Generate detailed itinerary using LLM
        itinerary_prompt = self._create_itinerary_prompt(
            primary_dest, hotels, attractions, days, user_preferences
        )
        
        detailed_itinerary = self._generate_with_llm(itinerary_prompt)
        
        # Compile complete response
        return {
            "destination": primary_dest,
            "alternative_destinations": destinations[1:],
            "recommended_hotels": hotels,
            "attractions": attractions,
            "detailed_itinerary": detailed_itinerary,
            "total_estimated_cost": self._calculate_cost(primary_dest, hotels, attractions, days)
        }
    
    def _get_budget_category(self, budget: int) -> str:
        """Determine hotel category based on budget"""
        if budget < 100:
            return "Budget"
        elif budget < 200:
            return "Mid-range"
        else:
            return "Luxury"
    
    def _create_itinerary_prompt(self, destination, hotels, attractions, days, preferences):
        """Create prompt for LLM to generate detailed itinerary"""
        attractions_list = "\n".join([
            f"- {a['name']} ({a['category']}, {a['duration_hours']}h, ${a['price']})"
            for a in attractions
        ])
        
        prompt = f"""Create a detailed {days}-day itinerary for {destination['city']}, {destination['country']}.

Traveler interests: {', '.join(preferences.get('interests', ['general sightseeing']))}

Available attractions:
{attractions_list}

Create a day-by-day plan with morning, afternoon, and evening activities."""
        
        return prompt
    
    def _generate_with_llm(self, prompt: str, max_length: int = 400) -> str:
        """Generate text using fine-tuned LLM"""
        full_prompt = f"""### Instruction:
{prompt}

### Response:
"""
        inputs = self.tokenizer(full_prompt, return_tensors="pt").to(self.model.device)
        
        outputs = self.model.generate(
            **inputs,
            max_length=max_length,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=self.tokenizer.eos_token_id
        )
        
        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        response = response.split("### Response:")[-1].strip()
        return response
    
    def _calculate_cost(self, destination, hotels, attractions, days):
        """Calculate estimated total cost"""
        daily_budget = destination['avg_budget_per_day']
        hotel_cost = hotels[0]['price_per_night'] * days if hotels else 0
        attraction_cost = sum([a['price'] for a in attractions])
        
        total = (daily_budget * days) + hotel_cost + attraction_cost
        
        return {
            "daily_expenses": daily_budget * days,
            "accommodation": hotel_cost,
            "attractions": attraction_cost,
            "total": total,
            "currency": "USD"
        }

# Initialize the agent
travel_agent = TravelItineraryAgent(
    model=model,
    tokenizer=tokenizer,
    vectorstore=vectorstore,
    df_destinations=df_destinations,
    df_hotels=df_hotels,
    df_attractions=df_attractions
)

print("✓ Travel Itinerary Agent initialized successfully!")

## 6. Testing and Evaluation

Test the complete system with various user preferences and evaluate the quality of generated itineraries.

In [ ]:
# Test Case 1: Romantic getaway
print("=" * 80)
print("TEST CASE 1: Romantic Getaway")
print("=" * 80)

preferences_1 = {
    "interests": ["romance", "culture", "food"],
    "max_budget": 180,
    "days": 5
}

result_1 = travel_agent.generate_itinerary(preferences_1)

print(f"\n🌍 Recommended Destination: {result_1['destination']['city']}, {result_1['destination']['country']}")
print(f"📝 Description: {result_1['destination']['description']}")
print(f"\n🏨 Recommended Hotel: {result_1['recommended_hotels'][0]['name']}")
print(f"   Category: {result_1['recommended_hotels'][0]['category']}")
print(f"   Price: ${result_1['recommended_hotels'][0]['price_per_night']}/night")
print(f"   Rating: {result_1['recommended_hotels'][0]['rating']}/5.0")

print(f"\n🎯 Top Attractions:")
for i, attr in enumerate(result_1['attractions'][:5], 1):
    print(f"   {i}. {attr['name']} - {attr['category']} (${attr['price']})")

print(f"\n💰 Estimated Cost Breakdown:")
cost = result_1['total_estimated_cost']
print(f"   Daily Expenses: ${cost['daily_expenses']}")
print(f"   Accommodation: ${cost['accommodation']}")
print(f"   Attractions: ${cost['attractions']}")
print(f"   TOTAL: ${cost['total']} {cost['currency']}")

print(f"\n📋 Detailed Itinerary:")
print(result_1['detailed_itinerary'])

In [ ]:
# Test Case 2: Adventure travel
print("\n" + "=" * 80)
print("TEST CASE 2: Adventure Travel")
print("=" * 80)

preferences_2 = {
    "interests": ["adventure", "nature", "beach"],
    "max_budget": 100,
    "days": 6
}

result_2 = travel_agent.generate_itinerary(preferences_2)

print(f"\n🌍 Recommended Destination: {result_2['destination']['city']}, {result_2['destination']['country']}")
print(f"📝 Description: {result_2['destination']['description']}")

if result_2['recommended_hotels']:
    print(f"\n🏨 Recommended Hotel: {result_2['recommended_hotels'][0]['name']}")
    print(f"   Category: {result_2['recommended_hotels'][0]['category']}")
    print(f"   Price: ${result_2['recommended_hotels'][0]['price_per_night']}/night")

print(f"\n🎯 Top Attractions:")
for i, attr in enumerate(result_2['attractions'][:5], 1):
    print(f"   {i}. {attr['name']} - {attr['category']} (${attr['price']})")

print(f"\n💰 Total Estimated Cost: ${result_2['total_estimated_cost']['total']} USD")

print(f"\n📋 Detailed Itinerary:")
print(result_2['detailed_itinerary'])

In [ ]:
# Test Case 3: Cultural exploration
print("\n" + "=" * 80)
print("TEST CASE 3: Cultural Exploration")
print("=" * 80)

preferences_3 = {
    "interests": ["culture", "history", "temples"],
    "max_budget": 150,
    "days": 7
}

result_3 = travel_agent.generate_itinerary(preferences_3)

print(f"\n🌍 Recommended Destination: {result_3['destination']['city']}, {result_3['destination']['country']}")
print(f"📝 Description: {result_3['destination']['description']}")
print(f"\n⭐ Alternative Destinations:")
for alt in result_3['alternative_destinations']:
    print(f"   - {alt['city']}, {alt['country']}")

print(f"\n🎯 Top Attractions:")
for i, attr in enumerate(result_3['attractions'][:5], 1):
    print(f"   {i}. {attr['name']} - {attr['category']} (${attr['price']})")

print(f"\n💰 Total Estimated Cost: ${result_3['total_estimated_cost']['total']} USD")

print(f"\n📋 Detailed Itinerary:")
print(result_3['detailed_itinerary'])

## 7. Interactive Interface

Create an interactive interface for users to input their preferences and get personalized itineraries.

In [ ]:
def interactive_travel_planner():
    """Interactive function to get user input and generate itinerary"""
    print("\n" + "="*80)
    print("🌎 AI-POWERED TRAVEL ITINERARY GENERATOR 🌎")
    print("="*80)
    
    print("\nAvailable interests: beach, culture, adventure, food, history, art, shopping,")
    print("                     temples, nature, romance, luxury, wellness, nightlife")
    
    # Get user input
    interests_input = input("\nEnter your interests (comma-separated): ")
    interests = [i.strip() for i in interests_input.split(',')]
    
    days_input = input("Number of days for the trip: ")
    days = int(days_input) if days_input else 5
    
    budget_input = input("Maximum budget per day (USD): ")
    max_budget = int(budget_input) if budget_input else 150
    
    # Create preferences dictionary
    preferences = {
        "interests": interests,
        "days": days,
        "max_budget": max_budget
    }
    
    print("\n🔍 Generating your personalized itinerary...\n")
    
    # Generate itinerary
    result = travel_agent.generate_itinerary(preferences)
    
    if "error" in result:
        print(f"❌ {result['error']}")
        return
    
    # Display results
    print("\n" + "="*80)
    print("YOUR PERSONALIZED TRAVEL ITINERARY")
    print("="*80)
    
    print(f"\n🌍 DESTINATION: {result['destination']['city']}, {result['destination']['country']}")
    print(f"\n{result['destination']['description']}")
    print(f"\n📅 Duration: {days} days")
    print(f"🌤️  Best time to visit: {result['destination']['best_season']}")
    
    if result['alternative_destinations']:
        print(f"\n💡 Alternative destinations you might like:")
        for alt in result['alternative_destinations']:
            print(f"   • {alt['city']}, {alt['country']}")
    
    print(f"\n🏨 RECOMMENDED ACCOMMODATION:")
    for hotel in result['recommended_hotels'][:2]:
        print(f"\n   {hotel['name']}")
        print(f"   Category: {hotel['category']} | ${hotel['price_per_night']}/night | ⭐ {hotel['rating']}/5.0")
        print(f"   Amenities: {', '.join(hotel['amenities'])}")
    
    print(f"\n🎯 TOP ATTRACTIONS & ACTIVITIES:")
    for i, attr in enumerate(result['attractions'], 1):
        print(f"\n   {i}. {attr['name']}")
        print(f"      Type: {attr['category']} | Duration: {attr['duration_hours']}h | Cost: ${attr['price']} | ⭐ {attr['rating']}/5.0")
    
    print(f"\n💰 ESTIMATED COST BREAKDOWN:")
    cost = result['total_estimated_cost']
    print(f"   Daily expenses (food, transport, etc.): ${cost['daily_expenses']}")
    print(f"   Accommodation ({days} nights): ${cost['accommodation']}")
    print(f"   Attractions & activities: ${cost['attractions']}")
    print(f"   " + "-"*50)
    print(f"   TOTAL ESTIMATED COST: ${cost['total']} {cost['currency']}")
    print(f"   (Flights not included)")
    
    print(f"\n📋 DETAILED DAY-BY-DAY ITINERARY:")
    print(f"\n{result['detailed_itinerary']}")
    
    print("\n" + "="*80)
    print("Happy travels! 🧳✈️")
    print("="*80)

# Run interactive planner
print("Ready to plan your dream vacation!")
print("Run the cell below to start the interactive planner.")

In [ ]:
# Run this cell to start the interactive travel planner
interactive_travel_planner()

## 8. System Evaluation

Evaluate the performance of the travel itinerary generator across multiple metrics.

In [ ]:
# Evaluation metrics
def evaluate_system():
    """Evaluate the system on various criteria"""
    
    test_cases = [
        {
            "name": "Budget Backpacker",
            "preferences": {"interests": ["adventure", "nature"], "max_budget": 80, "days": 5}
        },
        {
            "name": "Luxury Traveler",
            "preferences": {"interests": ["luxury", "beach"], "max_budget": 400, "days": 5}
        },
        {
            "name": "Cultural Explorer",
            "preferences": {"interests": ["culture", "history", "art"], "max_budget": 150, "days": 6}
        },
        {
            "name": "Family Vacation",
            "preferences": {"interests": ["beach", "entertainment"], "max_budget": 200, "days": 7}
        },
        {
            "name": "Food & Culture",
            "preferences": {"interests": ["food", "culture"], "max_budget": 120, "days": 4}
        }
    ]
    
    results = []
    
    print("="*80)
    print("SYSTEM EVALUATION REPORT")
    print("="*80)
    
    for test_case in test_cases:
        print(f"\nTest Case: {test_case['name']}")
        print(f"Preferences: {test_case['preferences']}")
        
        result = travel_agent.generate_itinerary(test_case['preferences'])
        
        if "error" not in result:
            # Check if destination matches interests
            dest_tags = set(result['destination']['tags'])
            user_interests = set(test_case['preferences']['interests'])
            match_score = len(dest_tags & user_interests) / len(user_interests)
            
            # Check budget compliance
            daily_budget = result['destination']['avg_budget_per_day']
            budget_ok = daily_budget <= test_case['preferences']['max_budget']
            
            # Check if attractions are relevant
            num_attractions = len(result['attractions'])
            
            results.append({
                "test_case": test_case['name'],
                "destination": result['destination']['city'],
                "interest_match": match_score,
                "budget_compliant": budget_ok,
                "num_attractions": num_attractions,
                "total_cost": result['total_estimated_cost']['total']
            })
            
            print(f"✓ Destination: {result['destination']['city']}")
            print(f"✓ Interest Match: {match_score*100:.0f}%")
            print(f"✓ Budget Compliant: {budget_ok}")
            print(f"✓ Attractions Found: {num_attractions}")
        else:
            print(f"✗ Error: {result['error']}")
    
    # Summary statistics
    print("\n" + "="*80)
    print("SUMMARY STATISTICS")
    print("="*80)
    
    df_results = pd.DataFrame(results)
    
    print(f"\nTotal test cases: {len(test_cases)}")
    print(f"Successful recommendations: {len(results)}")
    print(f"Success rate: {len(results)/len(test_cases)*100:.1f}%")
    print(f"\nAverage interest match: {df_results['interest_match'].mean()*100:.1f}%")
    print(f"Budget compliance rate: {df_results['budget_compliant'].sum()/len(results)*100:.1f}%")
    print(f"Average attractions per itinerary: {df_results['num_attractions'].mean():.1f}")
    
    print("\n" + "="*80)
    
    return df_results

# Run evaluation
evaluation_results = evaluate_system()

# Display detailed results
print("\nDetailed Results:")
print(evaluation_results)

## 9. Conclusions and Future Improvements

### System Capabilities:
1. ✅ **LLM Fine-tuning**: Successfully fine-tuned model on travel guides and itineraries
2. ✅ **RAG Implementation**: Vector store retrieval for real-time information
3. ✅ **Intelligent Agent**: Personalized itinerary generation based on preferences
4. ✅ **Multi-criteria Matching**: Budget, interests, and duration-based recommendations
5. ✅ **Cost Estimation**: Comprehensive budget breakdown for trips

### Future Improvements:
1. **Real-time Data Integration**:
   - Connect to live APIs (booking.com, TripAdvisor, flight APIs)
   - Weather forecasts and seasonal events
   - Real-time pricing and availability

2. **Enhanced ML Models**:
   - Larger language models (Llama 2, Mistral)
   - Multi-modal inputs (image-based destination discovery)
   - User feedback loop for continuous improvement

3. **Additional Features**:
   - Multi-city itineraries
   - Transportation planning (flights, trains, local transit)
   - Restaurant recommendations
   - Visa and travel document information
   - Travel insurance suggestions

4. **Personalization**:
   - User profile management
   - Historical trip analysis
   - Collaborative filtering for recommendations

5. **Deployment**:
   - Web application with user interface
   - Mobile app integration
   - API service for third-party integrations

### Technical Achievements:
- ✅ Implemented PEFT (LoRA) for efficient fine-tuning
- ✅ Created vector database for semantic search
- ✅ Built end-to-end RAG pipeline
- ✅ Designed modular agent architecture
- ✅ Comprehensive evaluation framework

---

**This notebook demonstrates a complete AI-powered travel assistant system that can be extended and deployed for real-world applications.**

## 10. References and Resources

### Libraries and Frameworks:
- **Transformers**: https://huggingface.co/docs/transformers
- **LangChain**: https://python.langchain.com/docs/get_started/introduction
- **PEFT**: https://huggingface.co/docs/peft
- **ChromaDB**: https://www.trychroma.com/
- **FAISS**: https://github.com/facebookresearch/faiss

### Research Papers:
- LoRA: Low-Rank Adaptation of Large Language Models
- Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks
- Attention Is All You Need (Transformer architecture)

### Datasets (for production use):
- TripAdvisor Hotel Reviews Dataset
- Booking.com Reviews Dataset
- Travel Guide Corpus
- Wikivoyage Travel Data

---

**Project Created for Tourism and Hospitality AI Applications**

**Google Colab Compatible** | **Open Source** | **Educational Purpose**